# Spotify Hit/Flop Classification
Classify tracks from the 2010s using audio features and a TensorFlow/Keras multilayer perceptron. This educational project predicts dataset labels, not future commercial success.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(0)
tf.random.set_seed(0)


## Load the dataset
Place `dataset-of-10s.csv` next to this notebook.

In [ ]:
data = pd.read_csv('dataset-of-10s.csv')
data.head()


## Class balance

In [ ]:
data.target.value_counts()


## Shuffle tracks

In [ ]:
data = data.sample(frac=1).reset_index(drop=True)
data.head()


## Danceability and energy

In [ ]:
import seaborn as sns
plt.figure(figsize=(6,4))
sns.scatterplot(x='danceability', y='energy', hue='target', data=data)
plt.title("Danceability and energy by Hit/Flop label")
plt.show()


## Feature correlations

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

features_to_include = ['danceability', 'energy', 'loudness', 'speechiness',
                       'acousticness', 'instrumentalness', 'liveness',
                       'valence', 'tempo', 'duration_ms', 'target']

df = data[features_to_include].copy()

corr_matrix = df.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', center=0)
plt.title("Feature and target correlations")
plt.show()


## Track metadata

Keep titles and artists for the interactive test-set predictions.

In [ ]:
titles_and_artists = data[['track', 'artist']].copy()


## Select numerical features

In [ ]:
data.drop(["track","artist","uri"], axis=1, inplace=True)
unscaled_inputs = data.iloc[:, 0:-1]
target = data.iloc[:, -1]


## Standardization
Fit the scaler on training data only, after splitting, to prevent data leakage.

In [ ]:
# Scaling is performed after splitting in the next cell.


## Train, validation and test sets
Use an 80% / 10% / 10% split.

In [ ]:
samples_count = unscaled_inputs.shape[0]
train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = unscaled_inputs.iloc[:train_samples_count]
train_targets = target[:train_samples_count]

val_inputs = unscaled_inputs.iloc[train_samples_count:train_samples_count+validation_samples_count]
val_targets = target[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = unscaled_inputs.iloc[train_samples_count+validation_samples_count:]
test_targets = target[train_samples_count+validation_samples_count:]

print("Train :", len(train_inputs), "samples")
print("Validation :", len(val_inputs), "samples")
print("Test :", len(test_inputs), "samples")

scaler = StandardScaler()
train_inputs = scaler.fit_transform(train_inputs)
val_inputs = scaler.transform(val_inputs)
test_inputs = scaler.transform(test_inputs)
test_metadata = titles_and_artists.iloc[train_samples_count + validation_samples_count:].reset_index(drop=True)


## Train the neural network
Three hidden layers of 50 ReLU units, followed by two softmax outputs. Adam optimizer, sparse categorical cross-entropy and early stopping on validation loss.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

input_size = unscaled_inputs.shape[1]
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

batch_size = 300
max_epochs = 100

history = model.fit(
    train_inputs,
    train_targets,
    batch_size=batch_size,
    epochs=max_epochs,
    validation_data=(val_inputs, val_targets),
    verbose=2,
    callbacks=[early_stopping]
)


## Training and validation loss

In [ ]:
training_loss = history.history['loss']
validation_loss = history.history['val_loss']
epoch_count = range(1, len(training_loss) + 1)

plt.plot(epoch_count, training_loss, 'r--')
plt.plot(epoch_count, validation_loss, 'b-')
plt.legend(['Training Loss', 'Validation Loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


## Evaluate the test set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print(f"\nTest loss: {test_loss:.2f}. Accuracy: {test_accuracy*100:.2f}%")


## Explore test-set predictions
The displayed score is an uncalibrated softmax output, not a verified probability of commercial success.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

rank_input = widgets.BoundedIntText(value=0, min=0, max=len(test_inputs)-1, description="Track index:")
button = widgets.Button(description="Predict")
output = widgets.Output()

def on_button_click(_):
    with output:
        clear_output()
        rank = rank_input.value
        scores = model.predict(test_inputs[rank].reshape(1, -1), verbose=0)[0]
        predicted = int(scores.argmax())
        actual = int(test_targets.iloc[rank])
        track = test_metadata.iloc[rank]
        labels = {0: "Flop", 1: "Hit"}
        print(f"{track['track']} — {track['artist']}")
        print(f"Prediction: {labels[predicted]} (softmax score: {scores[predicted]:.2%})")
        print(f"Actual dataset label: {labels[actual]}")

button.on_click(on_button_click)
display(rank_input, button, output)
